Agents and State Machine
========================


The agents are represented as an agent class and are provided when the environment is instantiated. Because agents can have different properties it is helpful
to know how to access this information.

In [12]:
import inspect

from flatland.envs.agent_utils import EnvAgent
from flatland.envs.rail_env import RailEnv
from flatland.env_generation.env_generator import env_generator

In [26]:
env, _, _ = env_generator()
env

You can simply access the three main types of agent information in the following ways `agent = env.agents[handle]`

In [29]:
print(env.agents[0])

EnvAgent(
	handle=0,
	initial_position=(8, 8),
	initial_direction=2,
	position=None,
	direction=None,
	targets={((22, 9), <Grid4TransitionsEnum.NORTH: 0>), ((22, 9), <Grid4TransitionsEnum.SOUTH: 2>)},
	old_position=None,
	old_direction=None,
	target_entry_point=None,
	earliest_departure=134,
	latest_arrival=172,
	state_machine=TrainStateMachine(
	state=0,
	previous_state=None,
	st_signals=StateTransitionSignals(in_malfunction=False, earliest_departure_reached=False, stop_action_given=False, movement_action_given=False, target_reached=False, movement_allowed=False, new_speed_zero=False, action_valid=False, ready_to_depart=False)
),
	malfunction_handler=MalfunctionHandler(
	malfunction_down_counter=0,
	num_malfunctions=0,
),
	waypoints=[[Waypoint(position=(8, 8), direction=2)], [Waypoint(position=(22, 9), direction=0), Waypoint(position=(22, 9), direction=2)]],
	waypoints_earliest_departure=[134, None],
	waypoints_latest_arrival=[None, 172],
)


Agent basic information
-----------------------

Since `flatland 5.0.0`, an agent on the map is on its way from its current entry point to its next entry point: it has entered the cell of
`agent.current_entry_point` and leaves it through `agent.next_entry_point`, the entry point of the neighbouring cell. In terms of the graph (see
[Simulation Model](../../key-concepts/simulation_model)), the agent is on the edge `(current_entry_point, next_entry_point)`, and
`agent.speed_counter.distance` is its offset on that edge: `0` at cell entry, the cell is left when the distance reaches `1` (see the speed information
below). Off the map, both entry points are `None`.

All the agent in the initiated environment can be found in the env.agents class. Given the index of the agent you have access to:

- **Agent entry point:** `agent.current_entry_point` which returns the current `((row, column), direction)` of the agent, with zero-based row and column
  and the direction as an int representing the current orientation {0: North, 1: East, 2: South, 3: West}. It is `None` while the agent is off the map,
  i.e. before departure and after arrival if `remove_agents_at_target` is set. The position is `agent.current_entry_point[0]`, the direction
  `agent.current_entry_point[1]`.
- **Agent next entry point:** `agent.next_entry_point` which returns the `((row, column), direction)` at which the agent will enter the next cell. It is
  `None` exactly when `agent.current_entry_point` is `None`. Otherwise, it is a successor of the current entry point (`env.rail.is_successor`), chosen by
  the action given when the agent entered its current cell (see [Actions](actions)).
- **Agent initial entry point:** `agent.initial_entry_point` which returns the `((row, column), direction)` at which the agent enters the map.
- **Agent old entry point:** `agent.old_entry_point` which returns the current entry point before the last step, `None` if the agent was off the map
  then.
- **Agent targets:** `agent.targets` which returns the set of target entry points, i.e. the target cell with the directions in which the agent can
  arrive. `agent.target_entry_point` is the one the agent actually arrived at, `None` until the agent is done. With `remove_agents_at_target` (the
  default), `agent.current_entry_point` is already `None` after the step in which the agent arrives: read
  `agent.target_entry_point` to know where it arrived.
- **Entry point for any agent:** `virtual_entry_point(agent)` (from `flatland.envs.agent_utils`) returns `agent.target_entry_point` once the agent is
  done, `agent.current_entry_point` while it is on the map, and `agent.initial_entry_point` before departure. Use it in observations and predictions
  which need an entry point for agents off the map, too.
- **Agent state:** `agent.state`, when moving, `agent.state == TrainState.MOVING` indicates that the train is currently on the map and moving. Since
  `flatland 5.0.0`, `step()` no longer uses the state, and there are two of them: the derived state `agent.derived_state(elapsed_steps)`, which is not
  stored but computed on each call from the agent's entry points, speed, malfunction and earliest departure, and which `step()` returns as
  `info['state']`; and the state of the agent's state machine, which only an env wrapped with `RailEnvStateMachineWrapper` updates, once per step, with
  the state transitions of `flatland` before 5.0.0. Without the wrapper, `agent.state` is `agent.derived_state()` without `elapsed_steps`, which reports
  READY_TO_DEPART as WAITING. See "Derived state and state machine wrapper" below for when they differ.

Agent timetable information
-----------------------

In **Flat**land 3, agents have a time window within which they must start and reach their destination. The following properties specify the time window:

- **Earliest departure:** `agent.earliest_departure` specifies the earliest time step of the simulation at which the agent is allowed to depart.
  Since `flatland 5.0.0`, an agent given a moving action is on the map after step `max(earliest_departure, 1)`, one step earlier than before, and is
  READY_TO_DEPART one step earlier, too (never for `earliest_departure <= 1`, as it can depart in step 1 already).
- **Latest arrival:** `agent.latest_arrival` specifies the latest time step of the simulation before or at which the agent is expected to reach it's
  destination.
- **Waypoints**: generalize initial position target, earliest departure and latest arrival by allowing for intermediate stops. The lists all have length at least 2 containing initial position and target and time windows (earliest, latest) for passing at those points without penalty by the reward function, however intermediate stops can be added in the list:
    - `agent.waypoints`: one list of alternative `Waypoint(position, direction)`s per stop; the target lists the target cell once for every direction in which it can be reached
    - `agent.waypoints_earliest_departure`: `None` for last element as no departure at target
    - `agent.waypoints_latest_arrival`: `None` for first element as no arrival at initial position

Agent malfunction information
-----------------------

Similar to the speed data you can also access individual data about the
malfunctions of an agent. All data is available through
`agent.malfunction_handler` with:

- **`malfunction_down_counter`:** Indication how long the agent is still malfunctioning by an integer counting down at each time step. `0` means the agent is ok
  and can move. Since `flatland 5.0.0`, the counter is decremented at the start of `step()`: after a step, it is greater than 0 exactly while the agent
  is in MALFUNCTION or MALFUNCTION_OFF_MAP, and the agent stays blocked for `malfunction_down_counter - 1` more steps. It is one higher than in `flatland 4.x`
  after the same step; the total duration of a malfunction is unchanged.
- **`num_malfunctions`:** Number of malfunctions an agent have occurred for this agent so far

Agent speed information
-----------------------

Beyond the basic agent information we can also access more details about
the agents type by looking at `agent.speed_counter`:

- **Agent max speed:** `agent.speed_counter.max_speed` wich defines the max traveling speed when the agent is moving.
- **Agent speed:** `agent.speed_counter.speed` wich defines the traveling speed when the agent is moving. Since `flatland 5.0.0`, speed and distance are
  `None` while the agent is off the map, i.e. before departure and after removal at the target, and `info['speed']` is `None` then, too.
- **Agent distance:** When the speed of an agent is fractional, the agent stays in the same cell for more than one step, specifically till `agent.speed_counter.distance` has reached 1.0, the cell length (which is currently the same for all cells). At each time step the agent is moving, its distance is incremented by its speed before the step; when the distance reaches 1.0, the agent enters the next cell with the remainder as distance. Since `flatland 5.0.0`, a speed change applies from the next step on.



State Machine
--------------




Flatland 3 introduces a state machine for every agent that controls the behavior of the agent depending on the current state.

Since `flatland 5.0.0`, the state machine no longer controls the behaviour: `step()` neither reads nor writes `agent.state`. The behaviour follows from the
agent's entry points, speed, distance, malfunction counter and earliest departure, and the state is derived from them (`agent.derived_state()`).
The state machine is an optional wrapper: `RailEnvStateMachineWrapper(env)` runs the transitions below, i.e. the state transitions of `flatland` before
5.0.0 with the changes of 5.0.0 listed below, at the end of each `step()` and updates `agent.state`. Between steps, it agrees with the derived state.
Most factories return an env which does not update `agent.state`, see the section "Derived state and state machine wrapper" below.

The functionality is essentially the same as Flatland 2, this change is
primarily to help users who want to read and modify the code easily, it is a refactoring of `env.step` function in Flatland.

The goal is to provide a unified state machine which controls the state and transitions of the trains, rather than the older system which had a mix of
`agent.moving` and `agent.status`.

The possible states are `WAITING`, `READY_TO_DEPART`, `MALFUNCTION_OFF_MAP`, `MOVING`, `STOPPED`, `MALFUNCTION`, and `DONE`.

In [32]:
from flatland.envs.step_utils.states import TrainState
for s in TrainState:
    print(f"\t{s.name} = {s.value}")

	WAITING = 0
	READY_TO_DEPART = 1
	MALFUNCTION_OFF_MAP = 2
	MOVING = 3
	STOPPED = 4
	MALFUNCTION = 5
	DONE = 6




The figure below illustrates the working of the state machine.

![State Machine](../../assets/images/state_machine.drawio.png)

[//]: # (```mermaid)

[//]: # (%%{ init: { 'theme': 'base', 'themeVariables': { 'background': '#f4f4f4' } } }%%)

[//]: # (graph TB)

[//]: # (    START&#40;&#40;"&nbsp;"&#41;&#41; --> WAITING)

[//]: # (    READY_TO_DEPART -- <font color = red>in_malfunction</font> --> MALFUNCTION_OFF_MAP)

[//]: # (    READY_TO_DEPART -- <font color = green>valid_movement_action_given</font> --> MOVING)

[//]: # (    MALFUNCTION_OFF_MAP -- NOT <font color = green>in_malfunction</font><br/><font color = red>ready_to_depart</font><br/><font color = green>valid_movement_action_given</font> --> MOVING)

[//]: # (    MOVING -- NOT <font color = red>in_malfunction</font><br/><font color = green>target_reached</font> --> DONE)

[//]: # (    DONE --> END&#40;&#40;&#40;"&nbsp;"&#41;&#41;&#41;)

[//]: # (    subgraph Off Map States)

[//]: # (        direction TB)

[//]: # (        WAITING -- <font color = red>in_malfunction</font> --> MALFUNCTION_OFF_MAP)

[//]: # (        WAITING -- NOT <font color = green>in_malfunction</font><br/><font color = red>earliest_departure_reached</font> --> READY_TO_DEPART)

[//]: # (        MALFUNCTION_OFF_MAP -- NOT <font color = green>in_malfunction</font><br/><font color = red>earliest_departure_reached</font><br/>NOT &#40;<font color = red>ready_to_depart</font> AND <font color = green>valid_movement_action_given</font>&#41; --> READY_TO_DEPART)

[//]: # (        MALFUNCTION_OFF_MAP -- NOT <font color = green>in_malfunction</font><br/>NOT <font color = red>earliest_departure_reached</font> --> WAITING)

[//]: # (    end)

[//]: # ()

[//]: # (    subgraph On Map States)

[//]: # (        direction TB)

[//]: # (        MOVING -- <font color = red>in_malfunction</font> --> MALFUNCTION)

[//]: # (        MOVING -- NOT <font color = red>in_malfunction</font><br/>NOT <font color = green>target_reached</font><br/>&#40;<font color = red>stop_action_given</font> AND new_speed = = 0&#41; OR NOT <font color = green>movement_allowed</font> --> STOPPED)

[//]: # (        STOPPED -- <font color = red>in_malfunction</font> --> MALFUNCTION)

[//]: # (        STOPPED -- NOT <font color = red>in_malfunction</font><br/><font color = green>valid_movement_action_given</font> --> MOVING)

[//]: # (        MALFUNCTION -- NOT <font color = green>in_malfunction</font><br/><font color = green>valid_movement_action_given</font> --> MOVING)

[//]: # (        MALFUNCTION -- NOT <font color = green>in_malfunction</font><br/>NOT <font color = green>valid_movement_action_given</font> --> STOPPED)

[//]: # (    end)

[//]: # ()

[//]: # (    START:::StartState)

[//]: # (    WAITING:::OffMapState)

[//]: # (    READY_TO_DEPART:::OffMapState)

[//]: # (    MALFUNCTION_OFF_MAP:::OffMapState)

[//]: # (    MOVING:::OnMapState)

[//]: # (    STOPPED:::OnMapState)

[//]: # (    MALFUNCTION:::OnMapState)

[//]: # (    END:::EndState)

[//]: # (    classDef OffMapState font-style: italic, font-weight: bold, fill: yellow, color: black)

[//]: # (    classDef OnMapState font-style: italic, font-weight: bold, fill: green, color: black)

[//]: # (    classDef StartState font-style: italic, font-weight: bold, fill: black, color: black)

[//]: # (    classDef EndState font-style: italic, font-weight: bold, fill: black, color: black)

[//]: # (```)

[//]: # ()

[//]: # (`valid_movement_action_given` stands for `movement_action_given` and `movement_allowed`)

[//]: # ()

[//]: # (Legend:)

[//]: # ()

[//]: # (```mermaid)

[//]: # (%%{ init: { 'theme': 'base', 'themeVariables': { 'background': '#f4f4f4' } } }%%)

[//]: # (stateDiagram-v2)

[//]: # (    direction LR)

[//]: # (    state "On Map State" as OnMapState)

[//]: # (    state "Off Map State" as OffMapState)

[//]: # (    OffMapState:::OffMapState)

[//]: # (    OnMapState:::OnMapState)

[//]: # (    state "State 1" as State1)

[//]: # (    state "State 2" as State2)

[//]: # (    state "State 1" as State3)

[//]: # (    state "State 2" as State4)

[//]: # (    State1 --> State2: <font color=red>Stopping signal</font>)

[//]: # (    State3 --> State4: <font color=green>Moving signal</font>)

[//]: # (    classDef OffMapState font-style: italic, font-weight: bold, fill: yellow, color: black)

[//]: # (    classDef OnMapState font-style: italic, font-weight: bold, fill: green, color: black)

[//]: # (```)

> This feature was introduced in `flatland 3.0.0`. The transitions changed as follows:
>
> | Version | Transition | Change |
> |---|---|---|
> | `flatland 3.0.7` | MALFUNCTION → STOPPED | When the malfunction ends, any action other than a valid movement action stops the train; before, only `stop_action_given` or a movement conflict did. |
> | [`flatland 4.0.6`](https://github.com/flatland-association/flatland-rl/pull/163) | MALFUNCTION_OFF_MAP → STOPPED | Needs `movement_allowed` in addition to `stop_action_given`. |
> | [`flatland 4.0.6`](https://github.com/flatland-association/flatland-rl/pull/163) | MALFUNCTION_OFF_MAP → READY_TO_DEPART | Taken unless a movement or stop action is given and `movement_allowed`; before, unless a valid movement action or a stop action was given. |
> | [`flatland 4.0.6`](https://github.com/flatland-association/flatland-rl/pull/163) | MOVING → STOPPED | `NOT movement_allowed` replaces the movement conflict. |
> | [`flatland 4.0.6`](https://github.com/flatland-association/flatland-rl/pull/136) | MOVING → STOPPED | A stop action stops the train only once it has braked to `new_speed == 0`. |
> | [`flatland 4.2.4`](https://github.com/flatland-association/flatland-rl/pull/361) | MOVING → STOPPED, STOPPED → MOVING, MALFUNCTION → MOVING/STOPPED, MALFUNCTION_OFF_MAP → MOVING/STOPPED | `stop_action_given` and `movement_action_given` are taken from the action given, no longer from the preprocessed action; `movement_allowed` also requires a valid action. A movement action that is invalid on the rail now stops the train through `NOT movement_allowed` instead of through a stop action set by preprocessing. |
> | [`flatland 4.2.4`](https://github.com/flatland-association/flatland-rl/pull/353) | MOVING → DONE | `target_reached` compares the cell and direction with the agent's set of targets (every direction in which the target cell can be entered) instead of comparing the position with the target. No change for generated environments. |
> | [`flatland 5.0.0`](https://github.com/flatland-association/flatland-rl/pull/501) | MALFUNCTION_OFF_MAP → STOPPED | Removed: a train cannot enter the map stopped. |
> | [`flatland 5.0.0`](https://github.com/flatland-association/flatland-rl/pull/529) | MALFUNCTION_OFF_MAP → MOVING | Needs `ready_to_depart` instead of `earliest_departure_reached`; otherwise the train goes to READY_TO_DEPART (or WAITING before earliest departure). |
>
> WAITING → READY_TO_DEPART (`NOT in_malfunction` and `earliest_departure_reached`) is unchanged since `flatland 3.0.0`. `flatland 4.3.0` changed no transitions (the state machine is compiled with Cython since then, [#449](https://github.com/flatland-association/flatland-rl/pull/449)).

The principles we try to implement are as follows:

1. Only a single state transition is allowed per `env.step`, except for reaching the target: a train entering its target goes on to DONE in the same step.

2. The action preprocessing ,position changes, speed counters and malfunctions are controlled by the state. Since `flatland 5.0.0`, it is the other way
   round: the state is derived from the position, speed counter and malfunction.

3. All signals for transitioning between states are generated in one place, for easy readability.

## State descriptions

The purpose of each state is described below:

|        State        |             Map Position             | Description                                                                                                                                                                                                                   |
|:-------------------:|:------------------------------------:|:------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
|       WAITING       |                Off Map               | No actions allowed here, when current timestep is behind earliest departure.                                                                                                                                                  |
|   READY_TO_DEPART   |                Off Map               | Train is ready to move and will start at `initial_position` when moving action is provided.                                                                                                                                   |
| MALFUNCTION_OFF_MAP |                Off Map               | When a randomized malfunction occurs in an `off map` state. Actions provided during the malfunction are not stored: when the malfunction ends, the train enters the map as MOVING if it is ready to depart, a movement action is provided in that step and movement is allowed, and goes back to READY_TO_DEPART (or WAITING before earliest departure) otherwise.                                                  |
|       MOVING        |                On Map                | Indicates the train is moving, if speed is 1.0, the train changes position every timestep. Since `flatland 5.0.0`, in the step in which a STOPPED or MALFUNCTION train starts moving, it only gets its speed and moves from the next step on.                                                                                                                                    |
|       STOPPED       |                On Map                | Indicates the train is stopped, this can occur when the user provides a stop action, or the train tries to move into a cell that is occupied or to a cell which does not have a track. Since `flatland 5.0.0`, a train given a stop action at the end of its cell completes the crossing and stops in the next cell.                                        |
|     MALFUNCTION     |                On Map                | When a randomized malfunction occurs in an `on map` state. No movement can occur during a malfunction state. Actions provided during the malfunction are not stored: when the malfunction ends, the train goes to MOVING if a movement action is provided in that step and movement is allowed, and to STOPPED otherwise. |
|        DONE         | Depends on `remove_agents_at_target` | This is the terminal state which is activated when the target is reached.                                                                                                                                                     |

## State transition signals

The corresponding signals in the state machine are described below

| State Transition Signal      | Description                                                                                                                                                                            |
|------------------------------|----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `in_malfunction`             | Malfunction states start when in_malfunction is set to true                                                                                                                            |
| `earliest_departure_reached` | Earliest departure reached in the next step (`earliest_departure <= elapsed_steps + 1`), drives WAITING → READY_TO_DEPART one step ahead of the departure.                               |
| `ready_to_depart`            | Train is allowed to depart in this step (`earliest_departure <= elapsed_steps`). Since `flatland 5.0.0`.                                                                                |
| `stop_action_given`          | Stop Action Given - User provided a stop action. Since `flatland 4.2.4`, it is no longer changed by action preprocessing (before, a moving action into an invalid position became a stop action). |
| `movement_action_given`      | Movement action is provided.                                                                                                                                                           |
| `target_reached`             | Target position is reached                                                                                                                                                             |
| `movement_allowed`           | `action_valid` and no conflict with other trains ("resource check"). Since `flatland 4.2.4`, it includes the validity of the action (before, an invalid action was turned into a stop action by preprocessing). Since `flatland 5.0.0`, granted optimistically when a STOPPED or MALFUNCTION train is given a movement action: it goes to MOVING and, if its way is still blocked, back to STOPPED in the next step. |
| `action_valid`               | The action is valid: inside the cell, or at the end of the cell with a valid transition. Since `flatland 5.0.0`.                                                                       |
| `new_speed_zero`             | Whether the new speed is zero: `MOVE_FORWARD` means accelerate and `STOP_MOVING` means braking. `MOVE_LEFT` and `MOVE_RIGHT` do not accelerate.                                        |

## Derived state and state machine wrapper

Since `flatland 5.0.0`, an agent has two states:

- **Derived state:** `agent.derived_state(elapsed_steps)` is not stored, nor precomputed by `step()`: each call derives it from the agent's attributes
  as they are at the time of the call. `step()` itself only uses these attributes, never a state.
- **State machine state:** an env wrapped with `RailEnvStateMachineWrapper` runs the state machine of every agent once per step, with the state
  transitions of `flatland` before 5.0.0 (see the changes above). It runs at the very end of `step()`, after the observations, rewards, dones and
  `info` are computed, just before the wrapped `step()` returns, and stores the result in `agent.state` and `agent.state_machine`.

They are available in the following ways:

- **`info['state']`:** `reset()` and `step()` return `agent.derived_state(env._elapsed_steps)` for every agent in `info['state']`, whether the env is
  wrapped or not. This is the reliable source of the state.
- **`agent.derived_state(elapsed_steps=None, in_malfunction=None)`:** derives the state from the agent's attributes only:
    - `DONE` once `agent.target_entry_point` is set,
    - off the map: `MALFUNCTION_OFF_MAP` while in malfunction, `READY_TO_DEPART` if `elapsed_steps >= 1` and `earliest_departure <= elapsed_steps + 1`,
      and `WAITING` otherwise. Without `elapsed_steps`, `READY_TO_DEPART` is reported as `WAITING`.
    - on the map: `MALFUNCTION` while in malfunction, `MOVING` if the speed is greater than 0, and `STOPPED` otherwise.
- **`agent.state`:** the state of the agent's state machine if the env is wrapped (see below), `agent.derived_state()` otherwise.
- **`RailEnvStateMachineWrapper(env, skip_state_machine_update=False)`** (from `flatland.envs.rail_env_state_machine_wrapper`): patches `env` in place
  and returns the same env, which then keeps the state machine state, see above. Wrapping an env which is already wrapped does not wrap it twice, it
  only sets `skip_state_machine_update`. With `skip_state_machine_update=True`, the env stays wrapped, but the state machine is not updated.
- **Factories:** `env_generator()`, `RailEnvPersister.load_new()`, `GraphRailEnv.from_graph()` and `load_flatland_environment_from_file()` take
  `skip_state_machine_update`, which defaults to `True`, i.e. the env does not update `agent.state`. Pass `False` to get a wrapped env. The policy runner
  (`flatland-trajectory-generate-from-policy`, see [Evaluation](../evaluation)) has the option `--skip-state-machine-update`, which defaults to `True`,
  too; pass `False` for a policy which reads `agent.state`. An env created with `RailEnv(...)` or `GraphRailEnv.from_rail_env()` is not wrapped.

`agent.state` and `agent.derived_state(env._elapsed_steps)` (`info['state']`) differ depending on where they are read:

| read                                                                                                               | wrapped `agent.state`                                                                                                           | unwrapped `agent.state`                                     | `agent.derived_state(env._elapsed_steps)`, `info['state']` |
|:-------------------------------------------------------------------------------------------------------------------|:--------------------------------------------------------------------------------------------------------------------------------|:------------------------------------------------------------|:-----------------------------------------------------------|
| between steps: after `reset()` or `step()`                                                                         | the state after the last step, the same as the derived state (checked if `check_step_pre_post_conditions`)                      | the derived state, but READY_TO_DEPART reported as WAITING  | the state after the last step                              |
| in `step()`, after the agents have moved: rewards, `EffectsGenerator.on_episode_step_end`, observations, predictors | the state after the **previous** step                                                                                           | the derived state, but READY_TO_DEPART reported as WAITING  | the state after this step                                  |
| in `step()`, in `EffectsGenerator.on_episode_step_start`                                                           | the state after the previous step                                                                                               | the derived state, but READY_TO_DEPART reported as WAITING, and malfunction already of this step | malfunction already of this step, see below               |

For example, for `earliest_departure = 3` and `MOVE_FORWARD` in every step (see [Speed Profiles](speed_profiles)):

| `elapsed_steps` after `step()` | derived state, `info['state']` | wrapped `agent.state` in observations | wrapped `agent.state` after `step()` | unwrapped `agent.state` |
|:-------------------------------|:-------------------------------|:--------------------------------------|:-------------------------------------|:------------------------|
| 1                              | WAITING                        | WAITING                               | WAITING                              | WAITING                 |
| 2                              | READY_TO_DEPART                | WAITING                               | READY_TO_DEPART                      | WAITING                 |
| 3                              | MOVING                         | READY_TO_DEPART                       | MOVING                               | MOVING                  |
| 4                              | MOVING                         | MOVING                                | MOVING                               | MOVING                  |

Before `flatland 5.0.0`, the state machine ran inside `step()`, before the observations, so an observation builder reading `agent.state` saw the state
after the step. Since 5.0.0, it sees the state after the previous step on a wrapped env: read `agent.derived_state(self.env._elapsed_steps)` instead, as
the built-in observation builders and predictors do.

`agent.derived_state()` matches the state after the step if it is called after `step()` has moved the agents: in observations, in `info`, in
`EffectsGenerator.on_episode_step_end` and between steps. In `EffectsGenerator.on_episode_step_start`, the malfunction counter is already decremented for
the current step, while the entry points and the speed are still those of the previous step: an agent whose malfunction ends in this step is reported as
no longer in malfunction one step early. Pass the malfunction status at the end of the previous step as `in_malfunction` there, as
`ConditionalMalfunctionEffectsGenerator` does (see [Stochasticity](stochasticity)).

[//]: # (## Old state diagram)

[//]: # ()

[//]: # (![Flatland 3 State Machine]&#40;https://flatland-association.github.io/flatland-book/_images/Flatland_3_State_Machine.jpg&#41;)

[//]: # ()

[//]: # (### Differences with code of old diagram added in the new diagram above)

[//]: # ()

[//]: # (* `MALFUNCTION OFF MAP --> READY_TO_DEPART`: malfunction_counter_complete and earliest_departure_reached **and not valid_movement_action_given and not)

[//]: # (  stop_action_given**)

[//]: # (* `MOVING --> DONE`: **not in_malfunction** and target_reached)

[//]: # (* `MOVING --> STOPPED`: **not in_malfunction and not target_reached** and &#40;stop_action_given or movement_conflict&#41;)

[//]: # (* `MALFUNCTION --> STOPPED`: malfunction_counter_complete **and not in_malfunction and not valid_movement_action_given**)